In [164]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Dict, List
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
from pathlib import Path

In [165]:
# creating schema to get structured output

class foldersSchema(BaseModel):
    folders: List[str] = Field(
        description="List of folder names"
    )

class mappingsSchema(BaseModel):
    file_mappings : Dict[str, List[str]] = Field(
        description="dictionary for storing folder to file mappings"
    )

In [166]:
#Creating model instance
model = ChatOllama(model="phi3:mini")
folders_structured_model = model.with_structured_output(foldersSchema)
mappings_structured_model = model.with_structured_output(mappingsSchema)

In [167]:
#creating state
class OrganizeState(TypedDict):
    desk_items:list[str]
    possible_folders:list[str]
    files_mapping:dict[str,list]

In [168]:
# dirs = f"""project_final_report.docx\n
# budget_2024.xlsx\n
# presentation_ai.pptx\n
# dataset_sales.csv\n
# dataset_customers.csv\n
# holiday_photo1.jpg\n
# holiday_photo2.png\n
# profile_picture.png\n
# movie_trailer.mp4\n
# song_classical.mp3\n
# streetfighter.exe\n
# GTA5.exe\n
# python_script.py\n
# app_server.js\n
# index.html\n
# styles.css\n
# machine_learning_project\n
# web_dev_course_notes.pdf\n
# research_paper_ai.pdf\n
# resume_muhtasham.docx\n
# system_logs\n
# backup_2023.zip\n
# docker-compose.yml\n
# config.yaml\n
# notes.txt"""

In [169]:
# node for fetching desktop items

def fetch_desk_items(state: OrganizeState):
    desk_items = []
    path = Path("C:/Users/hp/desktop")
    
    for item in path.iterdir():
        if not item.name.startswith('.'):
            desk_items.append(str(item.name))
    print("node 1 executed")

    return {'desk_items' : desk_items}


In [170]:
# node for generating folder names

def gen_folders(state : OrganizeState):
    desk_items = state['desk_items']

    prompt = f"""
    What folders should these files be grouped into? Keep folder names generic.

    Example:
    Work
    Media
    Stydy 

    Files:
    {desk_items}

    Return the result using the required schema.
    """

    result = folders_structured_model.invoke(prompt).folders
    print("node 2 executed")

    return {'possible_folders' : result}
    

In [ ]:
# node for creating folder -> file mappings

def gen_mappings(state : OrganizeState):
    folders = state['possible_folders']
    files = state['desk_items']

    prompt = f"""
    You are organizing files on a desktop.
    
    Example:
    
    Files:
    song.mp3
    movie.mp4
    resume.docx
    steam.lnk
    python_script.py
    git
    
    Folders:
    Documents
    Media
    Software
    Development
    Games
    Other
    
    Correct mapping:

    Documents : [resume.docx ]
    Media : [song.mp3, movie.mp4]
    Software : [steam.lnk, git]
    Development : [python_script.py]
    
    
    
    Now organize the following files.
    
    Files:
    {files}
    
    Available folders:
    {folders}
    
    Rules:
    - Each file must go into exactly one folder
    - Do NOT invent new folders
    - Use "Other" if unsure
    
    Return the result using the schema where key must be name of the folder and value must be list of related files.
    """

    result = mappings_structured_model.invoke(prompt).file_mappings
    print("node 3 executed")

    return {
        'files_mapping' : result
    }

In [172]:
# node for creating folders and copying files

def organize(state : OrganizeState):
    files_mapping = state['files_mapping']

    for key, value in files_mapping.items():
        folder = Path(f'C:/Users/hp/desktop/{key}')

        if folder.exists():
            print("Folder exists..skipping...")
            continue
        
        folder.mkdir(exist_ok=True)

        for v in value:
            file = Path(f'C:/Users/hp/desktop/{v}')
            dest = f'{folder}/{file.name}'

            try:
                if file.exists():
                    file.rename(dest)

            except PermissionError:
                print(f'Skipping locked folder or file : {file.name}')
        print("node 4 executed")

    return state

In [173]:
# creating graph 

graph = StateGraph(OrganizeState)

graph.add_node('fetch_desk_items', fetch_desk_items)
graph.add_node('gen_folders', gen_folders)
graph.add_node('gen_mappings', gen_mappings)
graph.add_node('organize', organize)

graph.add_edge(START, 'fetch_desk_items')
graph.add_edge('fetch_desk_items', 'gen_folders')
graph.add_edge('gen_folders', 'gen_mappings')
graph.add_edge('gen_mappings', 'organize')
graph.add_edge('organize', END)

In [174]:
workflow = graph.compile()

In [175]:
initial_state = {'desk_items': [], 'folders' : [], 'files_mapping' : []}
final_state = workflow.invoke(initial_state)
print(final_state)

node 1 executed
node 2 executed
node 3 executed
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
Folder exists..skipping...
{'desk_items': ['Blender 4.4.lnk', 'Brawlhalla.url', 'Combat Master.url', 'Discord.lnk',

In [176]:
print(final_state['files_mapping'])

{'Blender 4.4.lnk': ['Work/Software'], 'Brawlhalla.url': ['Work/Games'], 'Combat Master.url': ['Work/Games'], 'Discord.lnk': ['Work/Software'], 'Epic Games Launcher.lnk': ['Work/Software'], 'Fi.txt': ['Work/Other'], 'Free Download Manager.lnk': ['Work/Software'], 'Git Bash.lnk': ['Work/Development'], 'IntelliJ IDEA Community Edition 2024.2.3.lnk': ['Work/Development'], 'langraph-learning': ['Work/Study'], 'MongoDBCompass.lnk': ['Work/Development'], 'Muck.url': ['Work/Games'], 'Muhammad-Muhtasham-Resume.pdf': ['Work/Documents'], 'MuhammadMuhtasham_CoverLetter.pdf': ['Work/Documents'], 'Muhammad_Muhtasham.docx': ['Work/Documents'], 'osu!.lnk': ['Work/Games'], 'Postman.lnk': ['Work/Software'], 'Proton VPN.lnk': ['Work/Software'], 'Resume.docx': ['Work/Documents'], 'Roblox Player.lnk': ['Work/Games'], 'Roblox Studio.lnk': ['Work/Games'], 'Starting-Out-With-Java (2).pdf': ['Work/Study'], 'StarUML.lnk': ['Work/Other'], 'Steam.lnk': ['Work/Games'], 'text.txt': ['Work/Other'], 'TLauncher.lnk':